# Provision a new LLM Wiki project

This notebook does two things: (1) create a new project's schema and register it in the catalog, and (2) deploy/redeploy the Streamlit-in-Snowflake app. It intentionally does not sync, ingest, or index documents — that happens later, on demand, from the app's Data Sources page.

Run top to bottom. The catalog-setup cell only needs to run once ever; the project-creation cell once per new project; the deploy cell any time you change the app code or add a project.

In [ ]:
import sys
sys.path.insert(0, '../python')
from snowflake_session import get_session

session = get_session()
session.sql("USE ROLE ADVANCEDANALYTICS").collect()
session.sql("USE WAREHOUSE MTMWH02").collect()
session.sql("USE DATABASE MEDSOCMS").collect()
print('Connected.')

## One-time only: create the shared catalog schema
Skip this cell if `MEDSOCMS.APP_CATALOG` already exists (i.e. this isn't the first project).

In [ ]:
from utils.sql_script import run_sql_file

count = run_sql_file(session, '../sql/00_setup_catalog.sql')
print(f'Catalog schema ready ({count} statements executed).')

## Create your project
Fill in the values below and run. Skip this cell if the project already exists (re-running errors on the duplicate project_code).

In [ ]:
PROJECT_CODE = 'ORG_MM_CHAT'
PROJECT_NAME = 'ORG - Meeting Minutes - Chat'
DESCRIPTION = 'Chatbot to query the Meeting Minutes of OCMS Review Group (ORG)'
SHAREPOINT_SITE_URL = 'https://metrotrains.sharepoint.com/sites/cabinet-mr4'
SHAREPOINT_DEFAULT_FOLDER = 'https://metrotrains.sharepoint.com/:f:/s/cabinet-mr4/IgB1xW6fxABYQqW7zhSCst56AR9xCdADTTa_g7N6grpMMto?e=s7I7WV'
CREATED_BY = 'Advanced Analytics User (Maddy)'
QUERY_WAREHOUSE = 'MTMWH02'
COMPUTE_POOL = ''
SEGMENTATION_PROFILE = 'ORG_MEETING_MINUTES'  # tuned for meeting-minutes structure; see index_builder.py

existing = session.sql(
    "SELECT COUNT(*) AS C FROM MEDSOCMS.APP_CATALOG.PROJECTS WHERE PROJECT_CODE = ?",
    params=[PROJECT_CODE],
).collect()[0]["C"]

if existing > 0:
    print(f"Project '{PROJECT_CODE}' already exists — skipping.")
else:
    result = session.sql(
        'CALL CREATE_PROJECT(?, ?, ?, ?, ?, ?, ?, ?)',
        params=[PROJECT_CODE, PROJECT_NAME, DESCRIPTION, SHAREPOINT_SITE_URL,
                SHAREPOINT_DEFAULT_FOLDER, CREATED_BY, QUERY_WAREHOUSE, COMPUTE_POOL],
    ).collect()
    print(result[0][0])
    session.sql(
        "UPDATE MEDSOCMS.APP_CATALOG.PROJECTS SET SEGMENTATION_PROFILE = ? WHERE PROJECT_CODE = ?",
        params=[SEGMENTATION_PROFILE, PROJECT_CODE],
    ).collect()
    print(f"Segmentation profile set to '{SEGMENTATION_PROFILE}'.")

## Deploy the Streamlit app
Safe to re-run any time — re-stages every file with `overwrite=True` and redeploys the app object with `CREATE OR REPLACE`. `environment.yml` is deliberately staged at the stage **root** (not nested under `streamlit/`), since Streamlit-in-Snowflake only reads it from there.

In [ ]:
import os

proj = session.sql(
    "SELECT * FROM MEDSOCMS.APP_CATALOG.PROJECTS WHERE PROJECT_CODE = ?",
    params=[PROJECT_CODE],
).collect()
if not proj:
    raise ValueError(f"No project found with code '{PROJECT_CODE}' — run Step 2 first.")
p = proj[0]

STREAMLIT_STAGE = f"MEDSOCMS.APP_CATALOG.{p['STREAMLIT_STAGE_NAME']}"
STREAMLIT_APP_NAME = f"MEDSOCMS.APP_CATALOG.{p['STREAMLIT_APP_NAME']}"
APP_QUERY_WAREHOUSE = p['QUERY_WAREHOUSE']

APP_COMPUTE_POOL = p['COMPUTE_POOL']
if not APP_COMPUTE_POOL or str(APP_COMPUTE_POOL).strip().lower() in ("", "none", "null"):
    APP_COMPUTE_POOL = None

session.sql(f"CREATE STAGE IF NOT EXISTS {STREAMLIT_STAGE}").collect()

# python/ is staged preserving its folder structure (python/config.py ->
# @stage/python/config.py), a sibling of app.py at the stage root.
#
# streamlit/ is FLATTENED to the stage root instead of preserving its own
# folder name (streamlit/app.py -> @stage/app.py, streamlit/pages/1_Chat.py
# -> @stage/pages/1_Chat.py) — a nested MAIN_FILE (e.g. 'streamlit/app.py')
# reliably fails to load on this account with 'Python Interpreter Error:
# TypeError: bad argument type for built-in operation' at bootstrap,
# regardless of the app's own code (confirmed by isolating a trivial
# nested test app with no pages/ folder and no risky imports — it failed
# identically, while the same content at the stage root worked). MAIN_FILE
# below is 'app.py', not 'streamlit/app.py', to match.
for root, _, files in os.walk("../python"):
    rel_root = os.path.relpath(root, "..")
    stage_dir = f"@{STREAMLIT_STAGE}/{rel_root}"
    for fname in files:
        if fname.endswith(".py"):
            session.file.put(f"{root}/{fname}", stage_dir,
                              auto_compress=False, overwrite=True)

for root, _, files in os.walk("../streamlit"):
    rel_root = os.path.relpath(root, "../streamlit")   # "." or "pages"
    stage_dir = f"@{STREAMLIT_STAGE}" if rel_root == "." else f"@{STREAMLIT_STAGE}/{rel_root}"
    for fname in files:
        if fname.endswith(".py"):
            session.file.put(f"{root}/{fname}", stage_dir,
                              auto_compress=False, overwrite=True)

# Remove stage files that no longer exist locally — session.file.put only
# ever stages/overwrites, it never deletes, so a file removed from the repo
# (or from an earlier deploy layout) would otherwise linger on the stage
# and, for anything under pages/, keep showing up in the app's navigation.
for stale in (
    "streamlit/app.py",             # pre-flattening nested layout
    "streamlit/pages/1_Chat.py",
    "streamlit/pages/2_Data_Sources.py",
    "streamlit/pages/3_Sync_Status.py",
    "streamlit/pages/4_New_Project.py",
    "pages/4_New_Project.py",       # New Project page removed from the app
):
    session.sql(f"REMOVE @{STREAMLIT_STAGE}/{stale}").collect()

# Stage EXACTLY ONE dependency manifest, matching the runtime this project is
# actually configured for. Staging both at once is ambiguous — Snowflake can
# pick up pyproject.toml even on warehouse runtime and attempt PyPI
# resolution, which fails without a PyPI External Access Integration.
if APP_COMPUTE_POOL:
    manifest_to_stage = "pyproject.toml"
    manifest_to_remove = "environment.yml"
else:
    manifest_to_stage = "environment.yml"
    manifest_to_remove = "pyproject.toml"

local_path = f"../streamlit/{manifest_to_stage}"
if os.path.exists(local_path):
    session.file.put(local_path, f"@{STREAMLIT_STAGE}",
                      auto_compress=False, overwrite=True)

session.sql(f"REMOVE @{STREAMLIT_STAGE}/{manifest_to_remove}").collect()

# FROM (not the legacy ROOT_LOCATION) — required on accounts where
# ROOT_LOCATION has been retired for new/replaced Streamlit apps.
create_stmt = f"""
    CREATE OR REPLACE STREAMLIT {STREAMLIT_APP_NAME}
      FROM '@{STREAMLIT_STAGE}'
      MAIN_FILE = 'app.py'
      QUERY_WAREHOUSE = {APP_QUERY_WAREHOUSE}
"""
# GRAPH_API_ACCESS_INTEGRATION is attached on every runtime, not just
# container: warehouse-runtime Streamlit apps need it too for the
# SharePoint tab's outbound Graph API calls (graph_client.py uses
# `requests` directly) — without it those calls are blocked regardless
# of COMPUTE_POOL. This was missed in the original deploy cell, which
# only ever attached PYPI_ACCESS_INTEGRATION, and only under container
# runtime.
if APP_COMPUTE_POOL:
    create_stmt += f"""
      RUNTIME_NAME = 'SYSTEM$ST_CONTAINER_RUNTIME_PY3_11'
      COMPUTE_POOL = '{APP_COMPUTE_POOL}'
      EXTERNAL_ACCESS_INTEGRATIONS = (GRAPH_API_ACCESS_INTEGRATION, PYPI_ACCESS_INTEGRATION)
    """
else:
    # RUNTIME_NAME set explicitly here too (not just under container
    # runtime) as a harmless belt-and-suspenders default — the actual
    # fix for the app-load TypeError turned out to be the flattened
    # stage layout above, not this.
    create_stmt += """
      RUNTIME_NAME = 'SYSTEM$WAREHOUSE_RUNTIME'
      EXTERNAL_ACCESS_INTEGRATIONS = (GRAPH_API_ACCESS_INTEGRATION)
    """

session.sql(create_stmt).collect()

print(f"Streamlit app deployed: {STREAMLIT_APP_NAME}")
print(f"  Warehouse: {APP_QUERY_WAREHOUSE}")
print(f"  Runtime:   {'container (' + APP_COMPUTE_POOL + ')' if APP_COMPUTE_POOL else 'warehouse'}")
print(f"  Manifest:  {manifest_to_stage} (removed {manifest_to_remove} if present)")

## If the app still won't load
Snowsight's "Something went wrong" card only shows the top-level exception, not a traceback — not enough to root-cause a generic `TypeError: bad argument type for built-in operation` (this is what caught the unresolvable `openpyxl` dependency, since removed — `xlsx_parser.py` now uses only the Python standard library). Run the cell below in *this* notebook (same warehouse-runtime Python environment the app runs in) to import every current app dependency one at a time and print a full traceback for whichever one actually fails.

In [ ]:
import sys, traceback

for p in ("../python", "../streamlit"):
    if p not in sys.path:
        sys.path.insert(0, p)

# Same import order app.py exercises, split out module by module so
# whichever one fails prints its own full traceback instead of one
# opaque top-level error. openpyxl is intentionally NOT tested here —
# it isn't a dependency anymore (xlsx_parser.py was rewritten to use
# only the Python standard library: zipfile + xml.etree.ElementTree),
# so it will always show FAILED regardless of whether the app loads.
# That's expected, not a bug.
modules_to_test = [
    "requests",
    "pandas",
    "config",
    "snowflake_session",
    "query_engine",
    "utils.cortex_client",
    "utils.graph_client",
    "utils.sql_utils",
    "utils.sql_script",
    "utils.logging_utils",
    "ingestion.xlsx_parser",
    "ingestion.file_ingest",
    "ingestion.sharepoint_ingest",
    "ingestion.index_builder",
]

for m in modules_to_test:
    try:
        __import__(m)
        print(f"OK    {m}")
    except Exception:
        print(f"FAILED {m}")
        traceback.print_exc()
        print()

print("\nPython:", sys.version)


## Next step
Open the Streamlit app, select this project from the sidebar, and go to **Data Sources** to add documents — either by uploading files directly or pointing at a SharePoint folder. No further notebook steps are needed.